In [ ]:
"""
Experiment 3.18: Paradox on Diagonal Nets
==========================================

HYPOTHESIS:
  The Muon Paradox (diverse weights, consistent losses) requires gauge symmetry.
  It should VANISH for diagonal networks where dim(gauge) = 0.

  Each layer is a diagonal matrix d_i in R^n. The product = element-wise product
  of diagonals. There is no O(n) gauge group -- every parameter is physical.

  "Muon" on a diagonal = Newton-Schulz ortho of diag(d) -> sign normalization,
  i.e. each element -> +/- 1.  This is coordinate-wise sign(G).

PROTOCOL:
  4-layer diagonal deep linear (n=32), 20 independent runs, 500 steps.
  Compare weight diversity ratio and loss std for Muon vs SGD.

PREDICTION:
  No paradox -- diversity ratios (func_div / weight_div) should be SIMILAR
  for Muon and SGD, because there are no gauge directions to explore.

SETUP:
  - Target: random diagonal d_target in R^n
  - Each layer: diagonal d_i in R^n  (stored as 1D vectors, not full matrices)
  - Forward: f(x) = diag(d_L * d_{L-1} * ... * d_1) @ x  (element-wise product)
  - Loss: 0.5 * ||f(X) - diag(d_target) @ X||^2 / N
  - Muon: Newton-Schulz on diag(g_i) -> sign(g_i) * ones  (orthogonal diagonal)
"""

# Experiment 3.18: The Muon Paradox on Diagonal Networks -- Does the Paradox Require Gauge Symmetry?

## Theoretical Motivation

The **Muon Paradox** is the empirical observation that Muon-optimized deep linear networks exhibit
*diverse final weights* across independent runs, yet converge to *nearly identical loss values and
input-output functions*. Under the gauge-theoretic interpretation of Muon, this paradox arises because
Muon's Newton-Schulz orthogonalization acts as a **gauge-fixing** mechanism: it projects gradients onto
the Stiefel manifold, which moves weights along gauge orbits (physically equivalent reparameterizations)
without changing the network's functional behavior.

### The Critical Prediction: No Gauge Group => No Paradox

A **diagonal deep linear network** provides the ideal null-hypothesis test bed. In a diagonal network:

- Each layer is parameterized by a vector $\mathbf{d}_i \in \mathbb{R}^n$ (the diagonal of what would be a full weight matrix).
- The end-to-end map is $f(\mathbf{x}) = \mathrm{diag}(\mathbf{d}_L \odot \mathbf{d}_{L-1} \odot \cdots \odot \mathbf{d}_1) \, \mathbf{x}$.
- The product $\mathbf{d}_L \odot \cdots \odot \mathbf{d}_1$ is an **element-wise** product of vectors.

Crucially, diagonal matrices commute, and the only orthogonal diagonal matrices are sign matrices
$\mathrm{diag}(\pm 1, \pm 1, \ldots, \pm 1)$. The continuous gauge group $O(n)$ that acts between layers
of a full-matrix network **collapses to the discrete group** $\{-1, +1\}^n$ for diagonal networks.
In the tangent-space sense, $\dim(\text{gauge}) = 0$ -- there are no continuous gauge directions to explore.

**Therefore:** If the Muon Paradox is genuinely a gauge phenomenon, it should **vanish** on diagonal networks.
Muon and SGD should produce comparable diversity ratios because there are no gauge orbits for Muon to traverse.

### What "Muon" Becomes on Diagonal Networks

Newton-Schulz iteration on a diagonal matrix $\mathrm{diag}(\mathbf{g})$ reduces to the scalar iteration:
$$x_{k+1} = \frac{3}{2} x_k - \frac{1}{2} x_k^3$$
which has fixed points at $x = \pm 1$ (and unstable fixed point at $x = 0$). This converges to
$\mathrm{sign}(g_i)$ for each component -- the diagonal analogue of projecting onto the orthogonal group.
The update becomes coordinate-wise sign normalization rather than full orthogonal projection.

### Experimental Design

We compare two network architectures under identical conditions:
1. **Diagonal network** (dim(gauge) = 0): 4-layer, $n = 32$, element-wise parameterization
2. **Full-matrix network** (dim(gauge) > 0): 4-layer, $n = 32$, dense matrix parameterization

For each, we run 20 independent initializations with both SGD and Muon, then measure:
- **Weight diversity**: pairwise $\ell_2$ distance between final weight configurations
- **Function diversity**: pairwise Frobenius distance between final input-output maps (normalized)
- **Paradox ratio**: $\rho = d_{\text{func}} / d_{\text{weight}}$ (low ratio for Muon vs SGD = paradox present)

## 1. Imports and Reproducibility

In [ ]:
import numpy as np
import os

In [ ]:
np.random.seed(42)

## 2. Experimental Configuration

The following hyperparameters define the experimental protocol. Key design choices:

- **DIM = 32**: Each diagonal layer lives in $\mathbb{R}^{32}$. A full-matrix layer would have $32 \times 32 = 1024$ parameters, but the diagonal restriction reduces this to 32 parameters per layer -- precisely eliminating the $O(32)$ inter-layer gauge freedom.
- **NUM_LAYERS = 4**: Sufficient depth to create a non-trivial optimization landscape with multiple equivalent factorizations (in the full-matrix case) while remaining computationally tractable.
- **NUM_INDEPENDENT_RUNS = 20**: Each optimizer gets 20 independent random initializations. The pairwise comparisons yield $\binom{20}{2} = 190$ distance measurements -- enough to reliably estimate diversity statistics.
- **NS_ITERS = 5**: Newton-Schulz iterations for the Muon orthogonalization. On diagonal matrices, this drives each component toward $\pm 1$; on full matrices, it converges to the nearest orthogonal matrix.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
BATCH_SIZE = 64
LR_MUON = 0.02
LR_SGD = 0.01
MOMENTUM = 0.9
NS_ITERS = 5
NUM_INDEPENDENT_RUNS = 20
NUM_TEST_INPUTS = 50

In [ ]:
print("=" * 90)
print("EXPERIMENTAL CONFIGURATION")
print("=" * 90)
print(f"  Network dimension (n):          {DIM}")
print(f"  Number of layers (L):           {NUM_LAYERS}")
print(f"  Diagonal params per layer:      {DIM}  (vs {DIM*DIM} for full matrix)")
print(f"  Total diagonal params:          {NUM_LAYERS * DIM}")
print(f"  Total full-matrix params:       {NUM_LAYERS * DIM * DIM}")
print(f"  Training steps per run:         {NUM_STEPS}")
print(f"  Batch size:                     {BATCH_SIZE}")
print(f"  Independent runs per optimizer: {NUM_INDEPENDENT_RUNS}")
print(f"  Pairwise comparisons:           {NUM_INDEPENDENT_RUNS * (NUM_INDEPENDENT_RUNS - 1) // 2}")
print(f"  Newton-Schulz iterations:       {NS_ITERS}")
print(f"  SGD momentum:                   {MOMENTUM}")
print(f"  Nominal LR (Muon / SGD):        {LR_MUON} / {LR_SGD}")
print()
print("GAUGE STRUCTURE COMPARISON:")
print(f"  Diagonal net gauge group:       {{+1,-1}}^n  (discrete, dim = 0)")
print(f"  Full-matrix gauge group:        O({DIM})^{NUM_LAYERS - 1}  (continuous, dim = {(NUM_LAYERS - 1) * DIM * (DIM - 1) // 2})")
print(f"  Ratio (gauge dims / total params): diagonal = 0/{NUM_LAYERS * DIM} = 0.000")
gauge_dims_full = (NUM_LAYERS - 1) * DIM * (DIM - 1) // 2
total_full = NUM_LAYERS * DIM * DIM
print(f"                                     full     = {gauge_dims_full}/{total_full} = {gauge_dims_full / total_full:.3f}")
print("=" * 90)

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

### Target and Input Data Generation

The target function is a fixed random diagonal matrix $\mathbf{d}_{\text{target}} \in \mathbb{R}^{32}$. The network must learn to factorize this diagonal into $L = 4$ layers such that $\mathbf{d}_1 \odot \mathbf{d}_2 \odot \mathbf{d}_3 \odot \mathbf{d}_4 = \mathbf{d}_{\text{target}}$.

For diagonal networks, this factorization is **unique up to sign flips**: given any solution $(\mathbf{d}_1, \ldots, \mathbf{d}_4)$, we can flip the sign of component $i$ in layer $k$ and compensate by flipping it in another layer. These discrete sign symmetries are the *only* gauge redundancy -- there is no continuous family of equivalent solutions as in the full-matrix case.

In [ ]:
# Fixed target and data
d_target = np.random.randn(DIM)  # target diagonal
X_data = np.random.randn(DIM, BATCH_SIZE) * 0.3
X_test = np.random.randn(DIM, NUM_TEST_INPUTS) * 0.3

In [ ]:
print("Target diagonal d_target statistics:")
print(f"  Shape:    {d_target.shape}")
print(f"  Mean:     {np.mean(d_target):.4f}")
print(f"  Std:      {np.std(d_target):.4f}")
print(f"  Min/Max:  [{np.min(d_target):.4f}, {np.max(d_target):.4f}]")
print(f"  ||d_target||_2 = {np.linalg.norm(d_target):.4f}")
print(f"\nTraining data X_data: shape = {X_data.shape}, ||X||_F = {np.linalg.norm(X_data, 'fro'):.4f}")
print(f"Test data X_test:     shape = {X_test.shape}, ||X||_F = {np.linalg.norm(X_test, 'fro'):.4f}")
print(f"\nNote: X scaled by 0.3 to keep gradients well-conditioned at initialization.")

## 3. Diagonal Network Architecture (dim(gauge) = 0)

### Mathematical Structure

A diagonal deep linear network with $L$ layers maps input $\mathbf{x} \in \mathbb{R}^n$ to:
$$f(\mathbf{x}) = \mathrm{diag}\!\left(\prod_{i=1}^{L} \mathbf{d}_i\right) \mathbf{x}$$
where $\prod$ denotes the **element-wise (Hadamard) product**.

**Key property -- no continuous gauge freedom:**
In a full-matrix network $W_L \cdots W_1$, inserting $Q Q^T = I$ between layers ($W_k Q$ and $Q^T W_{k-1}$) yields an equivalent parameterization for any orthogonal $Q$. For diagonal matrices, the only diagonal orthogonal matrices are sign matrices, so the gauge group is $\{-1,+1\}^n$ -- a **finite** group with no continuous degrees of freedom.

This means every parameter in a diagonal network is "physical" (observable through the network's input-output behavior), unlike full-matrix networks where most of the parameter space is gauge redundancy.

In [ ]:
def init_diag(num_layers, seed=42):
    """Initialize diagonal layers near ones for stability."""
    rng = np.random.RandomState(seed)
    diags = []
    for _ in range(num_layers):
        d = np.ones(DIM) + rng.randn(DIM) * 0.1
        diags.append(d.copy())
    return diags

In [ ]:
# Verify initialization structure
test_diags = init_diag(NUM_LAYERS, seed=42)
print("Initialization verification (seed=42):")
for i, d in enumerate(test_diags):
    print(f"  Layer {i}: mean={np.mean(d):.4f}, std={np.std(d):.4f}, "
          f"min={np.min(d):.4f}, max={np.max(d):.4f}")

# Show the product -- this is the initial end-to-end function
prod_init = np.ones(DIM)
for d in test_diags:
    prod_init *= d
print(f"\n  Initial product d_1 * d_2 * d_3 * d_4:")
print(f"    mean={np.mean(prod_init):.4f}, std={np.std(prod_init):.4f}")
print(f"    ||prod - d_target||_2 = {np.linalg.norm(prod_init - d_target):.4f}")
print(f"\n  Note: Initialized near ones + small noise => product starts near 1^4 = 1,")
print(f"  while d_target has std ~1. The optimization must reshape the product to match.")

In [ ]:
def forward_diag(diags, X):
    """Forward pass: multiply X by product of diagonals."""
    prod = np.ones(DIM)
    for d in diags:
        prod = prod * d  # element-wise product
    # Apply: diag(prod) @ X = prod[:, None] * X
    return prod[:, None] * X

In [ ]:
def compute_loss_diag(diags, X, target_diag):
    """Quadratic loss."""
    pred = forward_diag(diags, X)
    target_out = target_diag[:, None] * X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

### Gradient Computation: Chain Rule on Element-wise Products

The gradient for diagonal layer $k$ has a clean closed form. Let $p_i = \prod_{j} d_{j,i}$ be the $i$-th element of the overall product. Then:

$$\frac{\partial \mathcal{L}}{\partial d_{k,i}} = \frac{\partial \mathcal{L}}{\partial p_i} \cdot \frac{p_i}{d_{k,i}}$$

where $\frac{\partial \mathcal{L}}{\partial p_i} = (p_i - d_{\text{target},i}) \cdot \langle x_i^2 \rangle_{\text{batch}}$.

This is notable: **every gradient component is a scalar**, and gradients for different components $i$ are independent. There is no matrix structure for Newton-Schulz to exploit -- the orthogonalization degenerates to component-wise sign normalization.

In [ ]:
def compute_gradients_diag(diags, X, target_diag):
    """Backprop for diagonal network."""
    num_layers = len(diags)
    N = X.shape[1]

    # Product of all diagonals
    prod = np.ones(DIM)
    for d in diags:
        prod = prod * d

    # Error: (prod - target_diag) element-wise
    err = prod - target_diag  # (DIM,)

    # Loss = 0.5 * mean_over_samples(sum_i (err_i * x_i)^2)
    # dL/d_prod_i = err_i * mean(x_i^2 across batch)  ... but let me be precise.
    # pred = prod[:, None] * X
    # target_out = target_diag[:, None] * X
    # diff = (prod - target_diag)[:, None] * X
    # loss = 0.5 * mean_cols(sum_rows(diff^2))
    # dL/dprod_i = err_i * mean(X_i^2 across batch)

    # Compute mean(X_i^2) for each dimension
    x_sq_mean = np.mean(X ** 2, axis=1)  # (DIM,)
    dL_dprod = err * x_sq_mean  # (DIM,)

    # Gradient for each layer:
    # prod = d_1 * d_2 * ... * d_L (element-wise)
    # dprod/d_{k,i} = prod_i / d_{k,i}
    grads = []
    for k in range(num_layers):
        # grad_k_i = dL_dprod_i * prod_i / d_k_i
        with np.errstate(divide='ignore', invalid='ignore'):
            grad_k = dL_dprod * prod / (diags[k] + 1e-30)
        grads.append(grad_k)

    return grads

### Newton-Schulz on Diagonal Matrices: Degeneration to Sign Normalization

This is the heart of why the paradox should vanish. In the full-matrix case, Newton-Schulz iteration:
$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k X_k^T X_k$$

projects gradients onto the Stiefel manifold (set of orthonormal frames), which introduces motion along gauge orbits. For diagonal matrices, $X^T X = X^2$ (element-wise), so the iteration becomes:
$$x_{k+1} = \frac{3}{2} x_k - \frac{1}{2} x_k^3$$

This is a scalar cubic map with stable fixed points at $x = \pm 1$. After normalization and 5 iterations, each gradient component converges to $\pm 1$ -- equivalent to `sign(g)`. This is **normalized SGD** applied independently to each coordinate, not a genuine orthogonal projection that could explore gauge directions.

In [ ]:
def newton_schulz_diagonal(g_vec, num_iters=NS_ITERS):
    """
    Newton-Schulz orthogonalization of a diagonal matrix diag(g_vec).
    A diagonal orthogonal matrix has entries +/-1.
    NS iteration on diag(g) converges to diag(sign(g)).
    We can verify: for diagonal X, A = X^T X = X^2 (diagonal),
    and 1.5*X - 0.5*X*A = 1.5*X - 0.5*X^3. Fixed points: x_i = +/-1.
    """
    norm = np.linalg.norm(g_vec)
    if norm < 1e-12:
        return g_vec.copy()
    x = g_vec / norm
    for _ in range(num_iters):
        # For diagonal: X_{k+1} = 1.5 * X_k - 0.5 * X_k^3
        x = 1.5 * x - 0.5 * x ** 3
    return x

In [ ]:
# Demonstrate NS diagonal convergence to sign(g)
test_g = np.array([0.5, -0.3, 1.2, -0.8, 0.01, -2.0])
ns_result = newton_schulz_diagonal(test_g, num_iters=NS_ITERS)
sign_result = np.sign(test_g)

print("Newton-Schulz on diagonal gradient -- convergence to sign(g):")
print(f"  Input g:       {test_g}")
print(f"  NS(g):         {np.round(ns_result, 6)}")
print(f"  sign(g):       {sign_result}")
print(f"  ||NS(g)| - 1|: {np.round(np.abs(np.abs(ns_result) - 1), 8)}")
print(f"\n  Conclusion: NS on diagonal converges to +/-1 per component (= sign normalization).")
print(f"  This is equivalent to normalized SGD, NOT orthogonal projection in any meaningful sense.")

## 4. Optimizer Definitions (SGD with Momentum vs Diagonal Muon)

Both optimizers use momentum ($\beta = 0.9$). The critical difference:

- **SGD**: $v_t = \beta v_{t-1} + g_t$; $\theta_t = \theta_{t-1} - \eta v_t$
  - Standard gradient descent with momentum. The gradient $g_t$ is used directly.

- **Muon (diagonal)**: $v_t = \beta v_{t-1} + \mathrm{NS}(g_t)$; $\theta_t = \theta_{t-1} - \eta v_t$
  - The gradient is first passed through Newton-Schulz, which for diagonal matrices yields $\mathrm{sign}(g_t)$.
  - This makes all gradient components equal in magnitude -- a form of adaptive coordinate normalization.
  - Crucially, on diagonal networks this normalization **cannot** induce gauge exploration because there are no continuous gauge directions.

In [ ]:
def sgd_step_diag(diags, velocities, grads, lr):
    for i in range(len(diags)):
        velocities[i] = MOMENTUM * velocities[i] + grads[i]
        diags[i] = diags[i] - lr * velocities[i]
    return diags, velocities

In [ ]:
def muon_step_diag(diags, velocities, grads, lr):
    for i in range(len(diags)):
        ortho_grad = newton_schulz_diagonal(grads[i])
        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
        diags[i] = diags[i] - lr * velocities[i]
    return diags, velocities

## 5. Adaptive Learning Rate Selection

To ensure a fair comparison, we automatically find the largest stable learning rate for each optimizer. A learning rate is "stable" if it does not cause the loss to diverge beyond $50\times$ the initial value within 100 warm-up steps. This prevents comparing a well-tuned SGD against a poorly-tuned Muon (or vice versa), which could create spurious differences in weight diversity.

In [ ]:
def find_stable_lr(optimizer_fn, candidates):
    for lr in candidates:
        np.random.seed(42)
        diags = init_diag(NUM_LAYERS)
        velocities = [np.zeros(DIM) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss_diag(diags, X_data, d_target)
        stable = True
        for step in range(100):
            grads = compute_gradients_diag(diags, X_data, d_target)
            diags, velocities = optimizer_fn(diags, velocities, grads, lr)
            loss = compute_loss_diag(diags, X_data, d_target)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return candidates[-1]

## 6. Convergence Basin Analysis -- The Core Measurement

This is the central measurement of the experiment. For each optimizer, we:

1. Run $N = 20$ independent training trajectories from different random initializations
2. Record the final weights, input-output function, and loss for each run
3. Compute **pairwise distances** between all $\binom{20}{2} = 190$ pairs of final states

The key metric is the **paradox ratio** $\rho = d_{\text{func}} / d_{\text{weight}}$:
- **Low $\rho$** (small function diversity relative to weight diversity) = weights are exploring gauge orbits (functionally equivalent reparameterizations) = **paradox present**
- **High $\rho$** (function diversity proportional to weight diversity) = weights are in physically distinct regions = **no paradox**

If gauge symmetry drives the paradox, then:
- Full-matrix Muon should have $\rho_{\text{Muon}} \ll \rho_{\text{SGD}}$ (paradox)
- Diagonal Muon should have $\rho_{\text{Muon}} \approx \rho_{\text{SGD}}$ (no paradox)

In [ ]:
def measure_convergence_basin(lr_sgd, lr_muon, num_runs, num_steps):
    """
    Run many independent initializations with each optimizer.
    Measure weight diversity, function diversity, loss diversity.
    """
    results = {}

    for opt_name, opt_fn, lr in [('SGD', sgd_step_diag, lr_sgd),
                                   ('Muon', muon_step_diag, lr_muon)]:
        final_diags_list = []
        final_functions = []
        final_losses = []

        for run_idx in range(num_runs):
            diags = init_diag(NUM_LAYERS, seed=1000 + run_idx)
            velocities = [np.zeros(DIM) for _ in range(NUM_LAYERS)]

            for step in range(num_steps):
                grads = compute_gradients_diag(diags, X_data, d_target)
                diags, velocities = opt_fn(diags, velocities, grads, lr)
                loss = compute_loss_diag(diags, X_data, d_target)
                if np.isnan(loss) or loss > 1e10:
                    break

            final_diags_list.append([d.copy() for d in diags])
            final_functions.append(forward_diag(diags, X_test).copy())
            final_losses.append(compute_loss_diag(diags, X_data, d_target))

        # Compute pairwise diversity
        n = len(final_diags_list)
        weight_dists = []
        func_dists = []
        for i in range(n):
            for j in range(i + 1, n):
                d_w = 0.0
                for k in range(NUM_LAYERS):
                    d_w += np.linalg.norm(final_diags_list[i][k] - final_diags_list[j][k]) ** 2
                weight_dists.append(np.sqrt(d_w))
                d_f = np.linalg.norm(final_functions[i] - final_functions[j], 'fro')
                d_f /= np.linalg.norm(X_test, 'fro')
                func_dists.append(d_f)

        results[opt_name] = {
            'weight_diversity_mean': np.mean(weight_dists),
            'weight_diversity_std': np.std(weight_dists),
            'func_diversity_mean': np.mean(func_dists),
            'func_diversity_std': np.std(func_dists),
            'loss_mean': np.mean(final_losses),
            'loss_std': np.std(final_losses),
            'losses': np.array(final_losses),
        }

    return results

## 7. Full-Matrix Reference Network (dim(gauge) > 0) -- Positive Control

To validate that our measurement protocol can detect the paradox when it should exist, we run the same experiment on a **full-matrix deep linear network**. This serves as a positive control.

For a 4-layer full-matrix network with $n = 32$:
- The inter-layer gauge group is $O(32)^3$ (one orthogonal group between each pair of adjacent layers)
- $\dim(\text{gauge}) = 3 \times \frac{32 \times 31}{2} = 1488$ continuous gauge degrees of freedom
- Total parameters: $4 \times 32^2 = 4096$
- **Gauge fraction**: $1488 / 4096 = 36.3\%$ of the parameter space is gauge redundancy

In this setting, Muon's Newton-Schulz orthogonalization projects gradients onto the Stiefel manifold, which actively explores gauge orbits. We expect to see the full paradox: high weight diversity with low function diversity under Muon.

In [ ]:
def init_weights_full(num_layers, seed=42):
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(DIM) + rng.randn(DIM, DIM) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward_full(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss_full(weights, X, W_target):
    pred = forward_full(weights, X)
    target_out = W_target @ X
    diff = pred - target_out
    return 0.5 * np.mean(np.sum(diff ** 2, axis=0))

In [ ]:
def compute_gradients_full(weights, X, W_target):
    num_layers = len(weights)
    N = X.shape[1]
    activations = [X.copy()]
    out = X.copy()
    for W in weights:
        out = W @ out
        activations.append(out.copy())
    target_out = W_target @ X
    delta = (activations[-1] - target_out) / N
    grads = []
    for i in range(num_layers - 1, -1, -1):
        G = delta @ activations[i].T
        grads.insert(0, G)
        if i > 0:
            delta = weights[i].T @ delta
    return grads

In [ ]:
def newton_schulz_full(G, num_iters=NS_ITERS):
    norm = np.linalg.norm(G, ord='fro')
    if norm < 1e-12:
        return G
    X = G / norm
    for _ in range(num_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def measure_convergence_basin_full(lr_sgd, lr_muon, num_runs, num_steps):
    W_target_full = np.diag(d_target)  # Use same target, but as full matrix

    results = {}
    for opt_name, lr in [('SGD', lr_sgd), ('Muon', lr_muon)]:
        final_weights_list = []
        final_functions = []
        final_losses = []

        for run_idx in range(num_runs):
            weights = init_weights_full(NUM_LAYERS, seed=1000 + run_idx)
            velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]

            for step in range(num_steps):
                grads = compute_gradients_full(weights, X_data, W_target_full)
                if opt_name == 'SGD':
                    for i in range(NUM_LAYERS):
                        velocities[i] = MOMENTUM * velocities[i] + grads[i]
                        weights[i] = weights[i] - lr * velocities[i]
                else:
                    for i in range(NUM_LAYERS):
                        ortho_grad = newton_schulz_full(grads[i])
                        velocities[i] = MOMENTUM * velocities[i] + ortho_grad
                        weights[i] = weights[i] - lr * velocities[i]
                loss = compute_loss_full(weights, X_data, W_target_full)
                if np.isnan(loss) or loss > 1e10:
                    break

            final_weights_list.append([w.copy() for w in weights])
            final_functions.append(forward_full(weights, X_test).copy())
            final_losses.append(compute_loss_full(weights, X_data, W_target_full))

        n = len(final_weights_list)
        weight_dists = []
        func_dists = []
        for i in range(n):
            for j in range(i + 1, n):
                d_w = 0.0
                for k in range(NUM_LAYERS):
                    d_w += np.linalg.norm(final_weights_list[i][k] - final_weights_list[j][k], 'fro') ** 2
                weight_dists.append(np.sqrt(d_w))
                d_f = np.linalg.norm(final_functions[i] - final_functions[j], 'fro')
                d_f /= np.linalg.norm(X_test, 'fro')
                func_dists.append(d_f)

        results[opt_name] = {
            'weight_diversity_mean': np.mean(weight_dists),
            'weight_diversity_std': np.std(weight_dists),
            'func_diversity_mean': np.mean(func_dists),
            'func_diversity_std': np.std(func_dists),
            'loss_mean': np.mean(final_losses),
            'loss_std': np.std(final_losses),
        }

    return results

## 8. Execute the Experiment

We now run both the diagonal and full-matrix experiments. The execution sequence:
1. Find stable learning rates for diagonal SGD and diagonal Muon
2. Run 20 independent diagonal-network training trajectories for each optimizer
3. Find stable learning rates for full-matrix SGD and full-matrix Muon
4. Run 20 independent full-matrix training trajectories for each optimizer
5. Compare paradox ratios across architectures

In [ ]:
print("=" * 90)
print("Experiment 3.18: Paradox on Diagonal Nets (dim(gauge) = 0)")
print("=" * 90)
print(f"Setup: {NUM_LAYERS}-layer diagonal net (n={DIM}), {NUM_INDEPENDENT_RUNS} runs, {NUM_STEPS} steps")
print(f"Prediction: NO paradox (diversity ratios similar for Muon and SGD)")
print("=" * 90)

In [ ]:
# Find stable learning rates
print("\nFinding stable learning rates...")
lr_sgd = find_stable_lr(sgd_step_diag, [0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001])
lr_muon = find_stable_lr(muon_step_diag, [0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001])
print(f"  Diagonal SGD lr: {lr_sgd}")
print(f"  Diagonal Muon lr: {lr_muon}")

In [ ]:
print("\nInterpretation of learning rate selection:")
print(f"  The LR finder tests candidates from aggressive to conservative.")
print(f"  Muon (diagonal) uses sign-normalized gradients => each update component has unit magnitude.")
print(f"  SGD uses raw gradients => update magnitude depends on gradient scale.")
print(f"  Selected LRs: SGD={lr_sgd}, Muon={lr_muon}")
if lr_muon >= lr_sgd:
    print(f"  Muon tolerates equal or higher LR -- sign normalization provides implicit stability.")
else:
    print(f"  SGD tolerates higher LR here -- the gradient magnitudes happen to be well-scaled.")

In [ ]:
# Run diagonal net basin analysis
print("\n--- DIAGONAL NET (dim(gauge) = 0) ---")
diag_results = measure_convergence_basin(lr_sgd, lr_muon, NUM_INDEPENDENT_RUNS, NUM_STEPS)

In [ ]:
for opt_name in ['SGD', 'Muon']:
    r = diag_results[opt_name]
    print(f"  {opt_name}: loss={r['loss_mean']:.6e} +/- {r['loss_std']:.6e}, "
          f"d_weight={r['weight_diversity_mean']:.6f}, d_func={r['func_diversity_mean']:.6f}")

In [ ]:
# Detailed diagonal results interpretation
print("\nDiagonal Net Interpretation:")
sgd_r = diag_results['SGD']
muon_r = diag_results['Muon']
wd_ratio = muon_r['weight_diversity_mean'] / (sgd_r['weight_diversity_mean'] + 1e-30)
fd_ratio = muon_r['func_diversity_mean'] / (sgd_r['func_diversity_mean'] + 1e-30)
print(f"  Weight diversity ratio (Muon/SGD): {wd_ratio:.4f}")
print(f"  Func diversity ratio (Muon/SGD):   {fd_ratio:.4f}")
print(f"  Loss convergence -- SGD:  {sgd_r['loss_mean']:.6e} +/- {sgd_r['loss_std']:.6e}")
print(f"                     Muon: {muon_r['loss_mean']:.6e} +/- {muon_r['loss_std']:.6e}")
print()
if abs(wd_ratio - fd_ratio) < 0.5 * max(wd_ratio, fd_ratio):
    print("  => Weight and function diversity scale proportionally -- NO paradox detected.")
    print("     This is consistent with dim(gauge) = 0: all weight differences are physical.")
else:
    print("  => Weight and function diversity diverge -- possible paradox signal!")
    print("     This would be UNEXPECTED for a gauge-free network.")

In [ ]:
# Run full-matrix reference
print("\n--- FULL MATRIX NET (dim(gauge) > 0) ---")
print("Finding full-matrix LRs...")

### 8b. Full-Matrix Positive Control

We now repeat the identical protocol on a full-matrix network. If our measurement framework is sound, the paradox should appear here: Muon will show high weight diversity (exploring gauge orbits via orthogonal projection) but low function diversity (all gauge-equivalent configurations produce the same input-output map).

In [ ]:
# Quick LR search for full-matrix
def find_lr_full(opt_name, candidates):
    W_target_full = np.diag(d_target)
    for lr in candidates:
        np.random.seed(42)
        weights = init_weights_full(NUM_LAYERS)
        velocities = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]
        initial_loss = compute_loss_full(weights, X_data, W_target_full)
        stable = True
        for step in range(100):
            grads = compute_gradients_full(weights, X_data, W_target_full)
            if opt_name == 'SGD':
                for i in range(NUM_LAYERS):
                    velocities[i] = MOMENTUM * velocities[i] + grads[i]
                    weights[i] = weights[i] - lr * velocities[i]
            else:
                for i in range(NUM_LAYERS):
                    ortho_grad = newton_schulz_full(grads[i])
                    velocities[i] = MOMENTUM * velocities[i] + ortho_grad
                    weights[i] = weights[i] - lr * velocities[i]
            loss = compute_loss_full(weights, X_data, W_target_full)
            if np.isnan(loss) or loss > initial_loss * 50:
                stable = False
                break
        if stable:
            return lr
    return candidates[-1]

In [ ]:
lr_sgd_full = find_lr_full('SGD', [0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001])
lr_muon_full = find_lr_full('Muon', [0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001])
print(f"  Full SGD lr: {lr_sgd_full}")
print(f"  Full Muon lr: {lr_muon_full}")

In [ ]:
full_results = measure_convergence_basin_full(lr_sgd_full, lr_muon_full,
                                                NUM_INDEPENDENT_RUNS, NUM_STEPS)

In [ ]:
for opt_name in ['SGD', 'Muon']:
    r = full_results[opt_name]
    print(f"  {opt_name}: loss={r['loss_mean']:.6e} +/- {r['loss_std']:.6e}, "
          f"d_weight={r['weight_diversity_mean']:.6f}, d_func={r['func_diversity_mean']:.6f}")

In [ ]:
# Detailed full-matrix results interpretation
print("\nFull-Matrix Net Interpretation:")
sgd_rf = full_results['SGD']
muon_rf = full_results['Muon']
wd_ratio_f = muon_rf['weight_diversity_mean'] / (sgd_rf['weight_diversity_mean'] + 1e-30)
fd_ratio_f = muon_rf['func_diversity_mean'] / (sgd_rf['func_diversity_mean'] + 1e-30)
print(f"  Weight diversity ratio (Muon/SGD): {wd_ratio_f:.4f}")
print(f"  Func diversity ratio (Muon/SGD):   {fd_ratio_f:.4f}")
print(f"  Loss convergence -- SGD:  {sgd_rf['loss_mean']:.6e} +/- {sgd_rf['loss_std']:.6e}")
print(f"                     Muon: {muon_rf['loss_mean']:.6e} +/- {muon_rf['loss_std']:.6e}")
print()
if wd_ratio_f > 1.5 and fd_ratio_f < 1.5:
    print("  => Classic paradox pattern: Muon has MORE weight diversity but LESS function diversity.")
    print("     This is the expected signature of gauge orbit exploration via NS orthogonalization.")
elif wd_ratio_f > 1.5:
    print("  => Muon has higher weight diversity. Function diversity also elevated --")
    print("     partial paradox (some gauge exploration, some functional difference).")
else:
    print("  => No clear paradox pattern in full matrix. This would challenge the gauge interpretation.")

## 9. Comparative Results Table

The **paradox ratio** $\rho = d_{\text{func}} / d_{\text{weight}}$ is the central diagnostic:
- $\rho \approx 1$: weight diversity translates directly to function diversity (no gauge exploration)
- $\rho \ll 1$: weights are diverse but functions are similar (gauge orbit exploration = paradox)

The **paradox strength** is defined as $\rho_{\text{SGD}} / \rho_{\text{Muon}}$:
- $\approx 1$: both optimizers produce similar paradox behavior (no differential gauge effect)
- $\gg 1$: Muon has disproportionately low function diversity relative to its weight diversity (Muon explores gauge orbits more than SGD)

In [ ]:
print(f"\n\n{'=' * 90}")
print("RESULTS: PARADOX RATIO = func_diversity / weight_diversity")
print(f"{'=' * 90}")
print(f"  (Lower ratio for Muon vs SGD = paradox present = gauge exploration)")
print()

In [ ]:
print(f"{'Network':<20} {'Optimizer':<10} {'Weight Div':>12} {'Func Div':>12} {'Ratio':>10} {'Loss Std':>14}")
print("-" * 80)

In [ ]:
# Diagonal
for opt_name in ['SGD', 'Muon']:
    r = diag_results[opt_name]
    wd = r['weight_diversity_mean']
    fd = r['func_diversity_mean']
    ratio = fd / wd if wd > 1e-15 else float('nan')
    print(f"{'Diagonal':<20} {opt_name:<10} {wd:>12.6f} {fd:>12.6f} {ratio:>10.6f} {r['loss_std']:>14.6e}")

In [ ]:
# Full matrix
for opt_name in ['SGD', 'Muon']:
    r = full_results[opt_name]
    wd = r['weight_diversity_mean']
    fd = r['func_diversity_mean']
    ratio = fd / wd if wd > 1e-15 else float('nan')
    print(f"{'Full Matrix':<20} {opt_name:<10} {wd:>12.6f} {fd:>12.6f} {ratio:>10.6f} {r['loss_std']:>14.6e}")

In [ ]:
# Compute the paradox strength = ratio_SGD / ratio_Muon (>1 means paradox present)
diag_ratio_sgd = diag_results['SGD']['func_diversity_mean'] / (diag_results['SGD']['weight_diversity_mean'] + 1e-30)
diag_ratio_muon = diag_results['Muon']['func_diversity_mean'] / (diag_results['Muon']['weight_diversity_mean'] + 1e-30)
full_ratio_sgd = full_results['SGD']['func_diversity_mean'] / (full_results['SGD']['weight_diversity_mean'] + 1e-30)
full_ratio_muon = full_results['Muon']['func_diversity_mean'] / (full_results['Muon']['weight_diversity_mean'] + 1e-30)

In [ ]:
diag_paradox = diag_ratio_sgd / (diag_ratio_muon + 1e-30)
full_paradox = full_ratio_sgd / (full_ratio_muon + 1e-30)

In [ ]:
print()
print(f"  Diagonal:    SGD ratio = {diag_ratio_sgd:.6f}, Muon ratio = {diag_ratio_muon:.6f}")
print(f"               Paradox strength (SGD/Muon) = {diag_paradox:.4f}")
print(f"  Full matrix: SGD ratio = {full_ratio_sgd:.6f}, Muon ratio = {full_ratio_muon:.6f}")
print(f"               Paradox strength (SGD/Muon) = {full_paradox:.4f}")

## 10. Formal Hypothesis Tests

Four tests, each probing a different aspect of the gauge-paradox connection:

| Test | Question | Pass Criterion |
|------|----------|---------------|
| **T1** | Does the full-matrix net show the paradox? | $\rho_{\text{Muon}} < \rho_{\text{SGD}}$ for full matrix |
| **T2** | Is the diagonal net paradox-free? | Paradox strength $\approx 1.0$ (within $[0, 2]$) |
| **T3** | Is the full-matrix paradox *stronger* than diagonal? | Full paradox strength > Diagonal paradox strength |
| **T4** | Does the diagonal net lack the full paradox *pattern*? | Not both (Muon higher weight div AND lower func div) |

All four passing would strongly confirm that **gauge symmetry is necessary for the Muon Paradox**.

In [ ]:
print(f"\n\n{'=' * 90}")
print("HYPOTHESIS TESTS")
print(f"{'=' * 90}")

In [ ]:
# T1: Full matrix should show paradox (Muon ratio < SGD ratio)
t1 = full_ratio_muon < full_ratio_sgd
print(f"\nT1: Full matrix shows paradox (Muon ratio < SGD ratio)?")
print(f"    Muon={full_ratio_muon:.6f} vs SGD={full_ratio_sgd:.6f}")
print(f"    --> {'PASS' if t1 else 'FAIL'}")

In [ ]:
# T2: Diagonal net should NOT show paradox (ratios similar)
# "Similar" means the paradox strength is close to 1.0 (within 2x)
t2 = abs(diag_paradox - 1.0) < 1.0  # paradox strength between 0 and 2
print(f"\nT2: Diagonal net does NOT show paradox (strength near 1.0)?")
print(f"    Paradox strength = {diag_paradox:.4f} (expect near 1.0)")
print(f"    --> {'PASS' if t2 else 'FAIL'}")

In [ ]:
# T3: Full matrix paradox should be STRONGER than diagonal
t3 = full_paradox > diag_paradox
print(f"\nT3: Full matrix paradox stronger than diagonal?")
print(f"    Full={full_paradox:.4f} vs Diagonal={diag_paradox:.4f}")
print(f"    --> {'PASS' if t3 else 'FAIL'}")

In [ ]:
# T4: In diagonal net, Muon should NOT have higher weight diversity than SGD
# (or if it does, it should also have proportionally higher func diversity)
diag_muon_higher_wd = diag_results['Muon']['weight_diversity_mean'] > diag_results['SGD']['weight_diversity_mean']
diag_muon_lower_fd = diag_results['Muon']['func_diversity_mean'] < diag_results['SGD']['func_diversity_mean']
t4_paradox_in_diag = diag_muon_higher_wd and diag_muon_lower_fd
print(f"\nT4: Diagonal net does NOT have the full paradox pattern?")
print(f"    (Muon higher weight div AND lower func div = paradox)")
print(f"    Muon higher weight div: {diag_muon_higher_wd}")
print(f"    Muon lower func div:    {diag_muon_lower_fd}")
print(f"    Full paradox pattern:   {t4_paradox_in_diag}")
print(f"    --> {'PASS (no paradox in diagonal)' if not t4_paradox_in_diag else 'FAIL (paradox found in diagonal!)'}")

In [ ]:
total_pass = sum([t1, t2, t3, not t4_paradox_in_diag])

## 11. Final Verdict and Scientific Interpretation

In [ ]:
print(f"\n\n{'=' * 90}")
print("FINAL VERDICT")
print(f"{'=' * 90}")

In [ ]:
print(f"""
  HYPOTHESIS: The Muon Paradox requires gauge symmetry (dim(gauge) > 0).
  It vanishes for diagonal networks where every parameter is physical.

  Tests passed: {total_pass}/4

  Diagonal net paradox strength: {diag_paradox:.4f} (expect ~1.0)
  Full matrix paradox strength:  {full_paradox:.4f} (expect >1.0)
""")

In [ ]:
if total_pass >= 3:
    print("  VERDICT: CONFIRMED -- Paradox requires gauge symmetry.")
    print("  The diagonal net (no gauge group) shows no paradox,")
    print("  while the full matrix net (O(n) gauge group) does.")
elif total_pass >= 2:
    print("  VERDICT: PARTIALLY CONFIRMED -- Some evidence for gauge requirement.")
else:
    print("  VERDICT: REJECTED -- Paradox appears even without gauge symmetry.")
    print("  This would mean the paradox mechanism is NOT purely gauge-theoretic.")

In [ ]:
print("=" * 90)

## 12. Conclusions and Broader Implications

### Summary of the Diagonal-Net Null Test

This experiment tested the **necessary condition** for the Muon Paradox: does it require continuous gauge symmetry? By comparing diagonal networks (dim(gauge) = 0) against full-matrix networks (dim(gauge) > 0) under identical measurement protocols, we can determine whether the paradox is a genuine gauge phenomenon or an artifact of other optimizer properties.

### What the Results Mean

**If confirmed (3-4 tests pass):**
The Muon Paradox is a *bona fide* gauge-theoretic phenomenon. Muon's Newton-Schulz orthogonalization generates weight diversity by exploring gauge orbits -- continuous families of reparameterizations that are physically equivalent (same input-output function). When these gauge orbits are eliminated by restricting to diagonal networks, the paradox vanishes. This provides strong evidence for the "Muon as RG Gauge Fixing" interpretation.

**If partially confirmed (2 tests pass):**
The gauge mechanism is likely one component of the paradox, but other factors (e.g., the implicit regularization from sign normalization, or the curvature landscape differences between optimizers) may also contribute.

**If rejected (0-1 tests pass):**
The paradox is NOT primarily gauge-theoretic, and we would need an alternative explanation -- perhaps related to the conditioning properties of Newton-Schulz orthogonalization, or the effect of gradient normalization on the optimization trajectory independent of gauge structure.

### Connection to the RG Gauge-Fixing Framework

In the renormalization group (RG) interpretation of deep linear networks:
- **Gauge transformations** correspond to inter-layer basis changes $W_k \to W_k Q$, $W_{k-1} \to Q^T W_{k-1}$
- **Physical observables** are the end-to-end product $W_L \cdots W_1$ and its singular value decomposition
- **Muon's NS orthogonalization** acts as a gauge-fixing condition, analogous to choosing a specific gauge in field theory (e.g., Coulomb gauge, Lorenz gauge)

The diagonal network strips away all gauge freedom, leaving only the "physical" degrees of freedom. If the paradox vanishes in this setting, it confirms that Muon's paradoxical behavior is entirely explained by gauge orbit exploration -- Muon finds solutions in different gauges (different weight-space locations on the same gauge orbit), which appear diverse in weight space but identical in function space.

### Limitations and Caveats

1. **Discrete sign symmetry remains**: Even diagonal networks have the discrete $\{-1,+1\}^n$ gauge group. The paradox test is specifically about *continuous* gauge directions (dim = 0 vs dim > 0).
2. **Linear networks only**: This experiment uses linear networks. Nonlinear activations break the exact gauge structure; whether the paradox persists partially in nonlinear networks is an open question (see Experiment H11).
3. **Fixed depth and width**: The strength of the paradox may depend on the ratio of gauge dimensions to total parameters, which changes with architecture.